<a href="https://colab.research.google.com/github/Jenna-learner/23521117_DS200_LAP3/blob/main/23521117_LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pyspark
from pyspark import SparkContext
import datetime
sc = SparkContext.getOrCreate()

Ô code 2: Xử lý Bài 1, Bài 2 và Bài 6

In [9]:
# Hàm xử lý đặc biệt đọc file movies.txt phòng trường hợp lỗi xuống dòng giữa tên phim
def load_clean_movies(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Sửa lỗi xuống dòng lỗi trong file gốc (như E.T. và L.A. Confidential)
    content = content.replace("\n the Extra-Terrestrial", " the Extra-Terrestrial")
    content = content.replace("\n Confidential", " Confidential")

    lines = [line.strip() for line in content.split("\n") if line.strip()]
    return sc.parallelize(lines)

print("Khởi tạo cấu hình xử lý dữ liệu thành công!")

Khởi tạo cấu hình xử lý dữ liệu thành công!


In [12]:
# Đọc dữ liệu sạch
ratings_raw = sc.textFile("ratings_1.txt,ratings_2.txt")
movies_raw = load_clean_movies("movies.txt")

def clean_line(line):
    return "userid" not in line.lower() and "movieid" not in line.lower() and len(line.strip()) > 0

ratings_clean = ratings_raw.filter(clean_line)
movies_clean = movies_raw.filter(clean_line)

# ==========================================
# BÀI 1: Điểm TB, tổng lượt vote & phim cao nhất
# ==========================================
print("--- BÀI 1 ---")
movies_map = movies_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), parts[1].strip()))
ratings_mapped = ratings_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[1].strip(), (float(parts[2]), 1)))

ratings_reduced = ratings_mapped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
movie_stats = ratings_reduced.mapValues(lambda v: (round(v[0] / v[1], 2), v[1]))
movie_stats_with_title = movie_stats.join(movies_map)

# Ít nhất >= 50 ratings
filtered_movies = movie_stats_with_title.filter(lambda x: x[1][0][1] >= 50)

if not filtered_movies.isEmpty():
    highest_rated_movie = filtered_movies.max(key=lambda x: x[1][0][0])
    print(f"Phim có điểm TB cao nhất (>=10 lượt vote): ID {highest_rated_movie[0]} - {highest_rated_movie[1][1]} | Điểm TB: {highest_rated_movie[1][0][0]} ({highest_rated_movie[1][0][1]} lượt vote)")
else:
    print("Không tìm thấy phim nào thỏa mãn điều kiện lọc số lượt đánh giá.")

print("\nThống kê danh sách phim (ID | Điểm TB | Số lượt đánh giá | Tên phim):")
for item in movie_stats_with_title.collect():
    print(f"ID: {item[0]} | Điểm TB: {item[1][0][0]} | Lượt: {item[1][0][1]} | Tên: {item[1][1]}")


# ==========================================
# BÀI 2: Phân tích theo thể loại phim
# ==========================================
print("\n--- BÀI 2 ---")
movie_genres_map = movies_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), [g.strip() for g in parts[2].split("|")]))
movie_ratings = ratings_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[1].strip(), float(parts[2])))

joined_genres_ratings = movie_ratings.join(movie_genres_map)
genre_rating_pairs = joined_genres_ratings.flatMap(lambda x: [(genre, (x[1][0], 1)) for genre in x[1][1]])

genre_stats = genre_rating_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                                .mapValues(lambda v: round(v[0] / v[1], 2))

print("Điểm trung bình theo từng thể loại phim:")
for genre, avg_rating in genre_stats.collect():
    print(f"- {genre}: {avg_rating}")


# ==========================================
# BÀI 6: Phân tích đánh giá theo Thời gian (Năm)
# ==========================================
print("\n--- BÀI 6 ---")
def timestamp_to_year(ts_str):
    try:
        return str(datetime.datetime.fromtimestamp(int(ts_str.strip())).year)
    except:
        return "Unknown"

year_rating_pairs = ratings_clean.map(lambda line: line.split(",")).map(lambda parts: (timestamp_to_year(parts[3]), (float(parts[2]), 1))).filter(lambda x: x[0] != "Unknown")
year_stats = year_rating_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                              .mapValues(lambda v: (v[1], round(v[0] / v[1], 2))) \
                              .sortByKey()

for year, stats in year_stats.collect():
    print(f"- Năm {year}: {stats[0]} lượt vote | Điểm TB: {stats[1]}")

--- BÀI 1 ---
Không tìm thấy phim nào thỏa mãn điều kiện lọc số lượt đánh giá.

Thống kê danh sách phim (ID | Điểm TB | Số lượt đánh giá | Tên phim):
ID: 1040 | Điểm TB: 3.61 | Lượt: 18 | Tên: Gladiator (2000)
ID: 1025 | Điểm TB: 4.06 | Lượt: 18 | Tên: The Terminator (1984)
ID: 1028 | Điểm TB: 3.5 | Lượt: 7 | Tên: Fight Club (1999)
ID: 1020 | Điểm TB: 3.67 | Lượt: 18 | Tên: E.T. the Extra-Terrestrial (1982)
ID: 1015 | Điểm TB: 4.36 | Lượt: 7 | Tên: Sunset Boulevard (1950)
ID: 1047 | Điểm TB: 3.86 | Lượt: 7 | Tên: The Social Network (2010)
ID: 1012 | Điểm TB: 4.0 | Lượt: 2 | Tên: Psycho (1960)
ID: 1050 | Điểm TB: 3.47 | Lượt: 18 | Tên: Mad Max: Fury Road (2015)
ID: 1037 | Điểm TB: 3.89 | Lượt: 18 | Tên: The Lord of the Rings: The Fellowship of the Ring (2001)
ID: 1013 | Điểm TB: 4.0 | Lượt: 17 | Tên: The Godfather: Part II (1974)
ID: 1039 | Điểm TB: 3.82 | Lượt: 11 | Tên: The Lord of the Rings: The Return of the King (2003)
ID: 1030 | Điểm TB: 3.14 | Lượt: 7 | Tên: The Silence of the La

Ô code 3: Xử lý Bài 3, Bài 4 và Bài 5

In [13]:
users_clean = sc.textFile("users.txt").filter(clean_line)

def get_age_group(age_str):
    try:
        age = int(age_str.strip())
        if age < 18: return "Under 18"
        elif age <= 24: return "18-24"
        elif age <= 34: return "25-34"
        elif age <= 44: return "35-44"
        elif age <= 49: return "45-49"
        elif age <= 55: return "50-55"
        else: return "56+"
    except:
        return "Unknown"

user_ratings_base = ratings_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), (parts[1].strip(), float(parts[2]))))

# ==========================================
# BÀI 3: Phân tích theo Giới tính
# ==========================================
print("--- BÀI 3 ---")
user_gender_map = users_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), parts[1].strip()))
joined_gender = user_ratings_base.join(user_gender_map)

movie_gender_pairs = joined_gender.map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))
movie_gender_stats = movie_gender_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda v: round(v[0] / v[1], 2))

movie_gender_rekeyed = movie_gender_stats.map(lambda x: (x[0][0], (x[0][1], x[1])))
movie_gender_with_title = movie_gender_rekeyed.join(movies_map)

print("Điểm trung bình của các phim phân theo giới tính (Tên phim | Giới tính | Điểm TB):")
for item in movie_gender_with_title.collect():
    print(f"- {item[1][1]} | Giới tính: {item[1][0][0]} | Điểm TB: {item[1][0][1]}")


# ==========================================
# BÀI 4: Phân tích theo Nhóm tuổi
# ==========================================
print("\n--- BÀI 4 ---")
user_age_map = users_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), get_age_group(parts[2])))
joined_age = user_ratings_base.join(user_age_map)

movie_age_pairs = joined_age.map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))
movie_age_stats = movie_age_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda v: round(v[0] / v[1], 2))

movie_age_rekeyed = movie_age_stats.map(lambda x: (x[0][0], (x[0][1], x[1])))
movie_age_with_title = movie_age_rekeyed.join(movies_map)

print("Điểm trung bình của các phim phân theo nhóm tuổi (Tên phim | Nhóm tuổi | Điểm TB):")
for item in movie_age_with_title.collect():
    print(f"- {item[1][1]} | Nhóm tuổi: {item[1][0][0]} | Điểm TB: {item[1][0][1]}")


# ==========================================
# BÀI 5: Phân tích theo Nghề nghiệp (Occupation)
# ==========================================
print("\n--- BÀI 5 ---")
occ_clean = sc.textFile("occupation.txt").filter(clean_line)
occ_map = occ_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), parts[1].strip())).collectAsMap()

user_occ_map = users_clean.map(lambda line: line.split(",")).map(lambda parts: (parts[0].strip(), parts[3].strip()))
joined_occ = user_ratings_base.join(user_occ_map)

def map_to_occ_name(x):
    occ_id = x[1][1]
    rating = x[1][0][1]
    occ_name = occ_map.get(occ_id, f"Job_ID_{occ_id}")
    return (occ_name, (rating, 1))

occ_rating_pairs = joined_occ.map(map_to_occ_name)
occ_stats = occ_rating_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda v: (v[1], round(v[0] / v[1], 2)))

print("Thống kê toàn diện điểm đánh giá theo Nghề nghiệp (Tên nghề: [Tổng lượt vote, Điểm TB]):")
for occ, stats in occ_stats.collect():
    print(f"- {occ}: {stats[0]} lượt vote | Điểm trung bình: {stats[1]}")

--- BÀI 3 ---
Điểm trung bình của các phim phân theo giới tính (Tên phim | Giới tính | Điểm TB):
- The Lord of the Rings: The Fellowship of the Ring (2001) | Giới tính: M | Điểm TB: 4.0
- The Lord of the Rings: The Fellowship of the Ring (2001) | Giới tính: F | Điểm TB: 3.8
- Mad Max: Fury Road (2015) | Giới tính: F | Điểm TB: 3.32
- Mad Max: Fury Road (2015) | Giới tính: M | Điểm TB: 4.0
- Sunset Boulevard (1950) | Giới tính: M | Điểm TB: 4.33
- Sunset Boulevard (1950) | Giới tính: F | Điểm TB: 4.5
- No Country for Old Men (2007) | Giới tính: M | Điểm TB: 3.92
- No Country for Old Men (2007) | Giới tính: F | Điểm TB: 3.83
- Fight Club (1999) | Giới tính: F | Điểm TB: 3.5
- Fight Club (1999) | Giới tính: M | Điểm TB: 3.5
- The Godfather: Part II (1974) | Giới tính: M | Điểm TB: 4.06
- The Godfather: Part II (1974) | Giới tính: F | Điểm TB: 3.94
- The Social Network (2010) | Giới tính: F | Điểm TB: 3.67
- The Social Network (2010) | Giới tính: M | Điểm TB: 4.0
- Lawrence of Arabia (1962